# 🍺 Roteirização de Bares — Comida di Buteco

Pipeline completo em 4 etapas:

| Etapa | O que faz |
|---|---|
| **1. Instalação** | Instala todas as dependências necessárias |
| **2. Scraper** | Coleta nome, endereço e horários de todos os bares do site |
| **3. Matriz de Distâncias** | Converte endereços em coordenadas e calcula distâncias reais por ruas (OSRM) |
| **4. Otimização** | Gera o roteiro semanal mais eficiente com OR-Tools e velocidade dinâmica por horário |

---
## ⚙️ Etapa 1 — Instalação de Dependências

Instala todas as bibliotecas usadas no projeto:
- **playwright** + **beautifulsoup4**: navegar no site e extrair HTML
- **geopy**: converter endereços de texto em coordenadas (lat/lon)
- **requests**: chamar a API OSRM para distâncias por ruas
- **ortools**: motor de otimização de rotas (Caixeiro Viajante)
- **openpyxl**: leitura e escrita de arquivos Excel

> ⏱️ Esta célula demora ~2 minutos na primeira execução.

In [2]:
!pip install -q playwright beautifulsoup4 geopy requests ortools openpyxl nest_asyncio
!playwright install chromium
!playwright install-deps chromium

print("\n✅ Todas as dependências instaladas com sucesso!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 21.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
175.4 MiB [] 0% 0.0s175.4 MiB [] 0% 43.4s175.4 MiB [] 0% 14.8s175.4 MiB [] 0% 13.3s175.4 MiB [] 0% 6.7s175.4 MiB [] 1% 4.2s175.4 MiB [] 2% 4.8s175.4 MiB [] 2% 5.1s175.4 MiB [] 3% 5.0s175.4 MiB [] 3% 5.2s175.4 MiB [] 3% 

---
## 🌐 Etapa 2 — Scraper: Coleta de Dados dos Bares

Acessa o site **comidadibuteco.com.br/butecos/campinas/** e coleta automaticamente:
- Nome e endereço de cada bar
- Horários de abertura e fechamento por dia da semana

**Como funciona:**
1. Abre o Chromium em modo headless (invisível)
2. Percorre todas as páginas da listagem coletando os links de detalhe
3. Acessa cada link e extrai os dados via BeautifulSoup
4. Salva tudo no arquivo `butecos_campinas.csv`

> ⏱️ Demora alguns minutos. O delay de 1.2s entre requisições é intencional para não sobrecarregar o servidor.

In [ ]:
import csv
import re
import time
import asyncio
import nest_asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

# nest_asyncio permite rodar código assíncrono dentro do loop do Colab
nest_asyncio.apply()

# ── Configuração ───────────────────────────────────────────────────────────────
BASE_URL    = "https://comidadibuteco.com.br"
LIST_URL    = "https://comidadibuteco.com.br/butecos/campinas/"
OUTPUT_FILE = "butecos_campinas.csv"
DELAY       = 1.2  # segundos entre requisições

# Mapa para normalizar nomes dos dias para o padrão do CSV
DIA_MAP = {
    "segunda": "segunda",       "segunda-feira": "segunda",
    "terça": "terça",           "terca": "terça",
    "terça-feira": "terça",     "terca-feira": "terça",
    "quarta": "quarta",         "quarta-feira": "quarta",
    "quinta": "quinta",         "quinta-feira": "quinta",
    "sexta": "sexta",           "sexta-feira": "sexta",
    "sábado": "sábado",         "sabado": "sábado",
    "domingo": "domingo",
}

CSV_COLUMNS = [
    "nome",
    "segunda-abre", "segunda-fecha",
    "terça-abre",   "terça-fecha",
    "quarta-abre",  "quarta-fecha",
    "quinta-abre",  "quinta-fecha",
    "sexta-abre",   "sexta-fecha",
    "sábado-abre",  "sábado-fecha",
    "domingo-abre", "domingo-fecha",
]


# ── Helpers de browser ─────────────────────────────────────────────────────────

async def new_context(browser):
    """Cria contexto simulando um usuário real para evitar bloqueios."""
    return await browser.new_context(
        user_agent=(
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        locale="pt-BR",
        viewport={"width": 1280, "height": 800},
    )

async def get_soup(url, browser):
    """Abre a URL no browser e retorna o HTML parseado pelo BeautifulSoup."""
    ctx  = await new_context(browser)
    page = await ctx.new_page()
    await page.goto(url, wait_until="networkidle", timeout=60000) # Increased timeout to 60 seconds
    await asyncio.sleep(1)
    html = await page.content()
    await ctx.close()
    return BeautifulSoup(html, "html.parser")


# ── Paginação e coleta de links ────────────────────────────────────────────────

def get_all_page_urls(soup_page1):
    """Descobre todas as URLs de página a partir dos links de paginação."""
    urls = [LIST_URL]
    for a in soup_page1.find_all("a", href=True):
        href = a["href"]
        if re.search(r"/butecos/campinas/page/\d+/", href) and href not in urls:
            urls.append(href)
    urls.sort(key=lambda u: int(re.search(r"/page/(\d+)/", u).group(1))
              if re.search(r"/page/(\d+)/", u) else 0)
    return urls

def get_detail_links(soup):
    """Retorna todos os links 'Detalhes' de uma página de listagem."""
    links = []
    for a in soup.find_all("a", href=True):
        if a.get_text(strip=True).lower() in ("detalhes", "detalhe"):
            href = a["href"]
            if not href.startswith("http"):
                href = BASE_URL + href
            if href not in links:
                links.append(href)
    return links


# ── Parsing de horários ────────────────────────────────────────────────────────

def normalizar_hora(hora):
    """Padroniza formatos de hora (ex: '18h', '18h30') para 'HH:MM'."""
    hora = hora.strip().lower().replace("h", ":").replace(".", ":")
    if hora.endswith(":"):
        hora += "00"
    partes = hora.split(":")
    if len(partes) == 1:
        return partes[0].zfill(2) + ":00"
    return partes[0].zfill(2) + ":" + partes[1].ljust(2, "0")

def parse_horario(texto):
    """
    Converte o texto de horários do site em dicionário {dia-abre, dia-fecha}.
    Suporta intervalos como 'Terça a Domingo: 18h - 23h'.
    """
    resultado = {}
    partes    = re.split(r"[|;\n]+", texto)
    ordem     = ["segunda", "terça", "quarta", "quinta", "sexta", "sábado", "domingo"]

    for parte in partes:
        parte = parte.strip()
        m = re.match(
            r"^(.+?):\s*(\d{1,2}[h:]\d{0,2})\s*[–\-]\s*(\d{1,2}[h:]\d{0,2})",
            parte, re.IGNORECASE,
        )
        if not m:
            continue
        dias_raw = m.group(1).strip().lower()
        abre     = normalizar_hora(m.group(2))
        fecha    = normalizar_hora(m.group(3))

        iv = re.match(r"(.+?)\s+a\s+(.+)", dias_raw)
        if iv:
            k1 = DIA_MAP.get(iv.group(1).strip())
            k2 = DIA_MAP.get(iv.group(2).strip())
            if k1 and k2:
                for dia in ordem[ordem.index(k1): ordem.index(k2) + 1]:
                    resultado[f"{dia}-abre"]  = abre
                    resultado[f"{dia}-fecha"] = fecha
        else:
            chave = DIA_MAP.get(dias_raw.replace("-feira", "").strip())
            if chave:
                resultado[f"{chave}-abre"]  = abre
                resultado[f"{chave}-fecha"] = fecha

    return resultado

def extrair_campo_p(soup, label):
    """Busca <p> com <b>label</b> e retorna o texto do campo (endereço ou horário)."""
    pattern = re.compile(label, re.I)
    for p in soup.find_all("p"):
        b = p.find("b")
        if b and pattern.search(b.get_text(strip=True)):
            valor = p.get_text(strip=True).replace(b.get_text(strip=True), "", 1).strip()
            if valor:
                return valor
    return ""


# ── Scrape de um bar ───────────────────────────────────────────────────────────

async def scrape_buteco(url, browser):
    """Acessa a página de detalhe de um bar e extrai nome, endereço e horários."""
    try:
        soup = await get_soup(url, browser)
    except Exception as e:
        print(f"  [ERRO ao acessar]: {e}")
        return None

    dados = {col: "" for col in CSV_COLUMNS}

    for tag in ["h1", "h2"]:
        el = soup.find(tag)
        if el:
            dados["nome"] = el.get_text(strip=True)
            break

    horario_txt = extrair_campo_p(soup, r"hor[aá]rio")
    if horario_txt:
        dados.update(parse_horario(horario_txt))

    return dados


# ── Execução principal (assíncrona) ────────────────────────────────────────────

async def main():
    print(f"Iniciando scraper — {LIST_URL}\n")

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)

        # Página 1: descobrir quantas páginas existem
        print("Carregando página 1...")
        soup1     = await get_soup(LIST_URL, browser)
        page_urls = get_all_page_urls(soup1)
        print(f"  Páginas encontradas: {len(page_urls)} → {page_urls}\n")

        # Coletar todos os links de detalhe em todas as páginas
        todos_links = []
        for i, purl in enumerate(page_urls, 1):
            soup_pag = soup1 if i == 1 else await get_soup(purl, browser)
            if i > 1:
                await asyncio.sleep(DELAY)
            links_pag = get_detail_links(soup_pag)
            print(f"  Página {i}: {len(links_pag)} butecos encontrados")
            todos_links += [lk for lk in links_pag if lk not in todos_links]

        print(f"\nTotal de butecos: {len(todos_links)}\n")

        # Scrape de cada buteco
        resultados = []
        for i, link in enumerate(todos_links, 1):
            print(f"[{i:>3}/{len(todos_links)}] {link}")
            dados = await scrape_buteco(link, browser)
            if dados:
                resultados.append(dados)
                print(f"          ✓ {dados.get('nome','?')} | terça abre: {dados.get('terça-abre','?')}")
            else:
                print(f"          ✗ falhou")
            await asyncio.sleep(DELAY)

        await browser.close()

    # Salvar CSV
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        writer.writeheader()
        writer.writerows(resultados)

    print(f"\n✅ {len(resultados)} butecos salvos em '{OUTPUT_FILE}'.")


asyncio.run(main())

Iniciando scraper — https://comidadibuteco.com.br/butecos/campinas/

Carregando página 1...
  Páginas encontradas: 6 → ['https://comidadibuteco.com.br/butecos/campinas/', 'https://comidadibuteco.com.br/butecos/campinas/page/2/', 'https://comidadibuteco.com.br/butecos/campinas/page/3/', 'https://comidadibuteco.com.br/butecos/campinas/page/4/', 'https://comidadibuteco.com.br/butecos/campinas/page/5/', 'https://comidadibuteco.com.br/butecos/campinas/page/6/']

  Página 1: 12 butecos encontrados
  Página 2: 12 butecos encontrados
  Página 3: 12 butecos encontrados
  Página 4: 12 butecos encontrados
  Página 5: 12 butecos encontrados
  Página 6: 1 butecos encontrados

Total de butecos: 61

[  1/61] https://comidadibuteco.com.br/buteco/1-poco-loco/
          ✓ 1 Poco Loco | terça abre: 16:00
[  2/61] https://comidadibuteco.com.br/buteco/bar-da-loira/
          ✓ Bar da Loira | terça abre: 11:00
[  3/61] https://comidadibuteco.com.br/buteco/bar-da-vila/
          ✓ Bar da Vila | terça abre: 1

---
## 📍 Etapa 3 — Matriz de Distâncias via OSRM

Prepara a base geográfica para a otimização:

1. **Geocoding híbrido**: localização informada como coordenadas diretas *ou* endereço de texto. Se for endereço, o **geopy/Nominatim** converte para lat/lon automaticamente.
2. **API OSRM**: calcula a distância real *por ruas* entre todos os pares de locais.
3. Resultado: matriz **N×N** de distâncias em km, salva em `matriz_distancias_bares.xlsx`.

> 📌 **Ponto de Saída**: base de onde começa e termina cada roteiro diário (Praça Bento Quirino, Campinas).
>
> ⚠️ O Nominatim tem limite de 1 req/segundo — o delay é automático.

In [ ]:
import requests
import time
import pandas as pd
from geopy.geocoders import Nominatim

# ── Lista de bares com localização ────────────────────────────────────────────
# Aceita coordenadas ('-22.90,-47.05') ou endereço de texto.
# Coordenadas são usadas diretamente; endereços passam pelo Nominatim.

bares = [
    "Ponto de Saida",

    "Bar da Loira",
    "Bar da Vila",
    "Bar do Baú",
    "Bar do Braz",
    "Bar do Cação",
    "Bar do Carioca",

    "Bar do Furlan",
    "Bar do Léo",
    "Bar do Luís",
    "Bar do Nicola",
    "Bar do Português",
    "Bar do Rafa",
    "Bar do Saulo",
    "Bar Dom Dama",
    "Bar Dona Flor",
    "Bar Du 6",
    "Bar Lumen",
    "Bar Seo Fabinho",
    "Bardellas",
    "Boró Botequim",
    "Boteco do Pezão",
    "Boteco do Wilma",
    "Boteco Sem Frescura",
    "Botelho's Bar",
    "Botequim de Barão",
    "Botiquim Pedro II",
    "Buteco do Chicão",

    "Cantina 85",
    "Casquinha Bar",
    "Cesário's Bar",
    "Collab Bar",
    "Deck 269",
    "Edu Bar de Minas",
    "Empório D' Primeira",
    "Eskina Bar",
    "Espaço In Bar",

    "Estação Aquidabã",
    "Estação Nova Europa",
    "F.B.I.",
    "Garten Bar",
    "Loschibar",
    "Maley Bistrô Bar",
    "Maria Maria",
    "Miúdo",

    "Pirajú Botequim",
    "Poeta Grill",
    "Quintal do Lindomar",
    "Quiosque da Horta",
    "Rei do Joelho",
    "Resenha dos Barbozas",
    "Santa Esquina",
    "Santo Espeto",
    "Seo Bar",
    "Seo Roma Bar e Restaurante",
    "Stadium Bar",
    "Vila's Bar",
]

dados_loc = [
    "-22.90620409,-47.07277313",   # Ponto de Saida — R. Dr. Pereira Lima, 85

    "-22.92359299,-47.10379216",   # Bar da Loira
    "-22.92108355,-47.03440566",   # Bar da Vila
    "-22.91691277,-47.08903562",   # Bar do Baú
    "-22.92006279,-47.09463124",   # Bar do Braz
    "-22.88077191,-47.05143675",   # Bar do Cação
    "-22.89423156,-47.07791188",   # Bar do Carioca

    "-22.90008651,-47.06682096",   # Bar do Furlan
    "-22.90982645,-47.09947021",   # Bar do Léo
    "-22.92215739,-47.07518852",   # Bar do Luís
    "-22.88290803,-47.05525329",   # Bar do Nicola
    "-22.90542140,-47.05285125",   # Bar do Português
    "-22.90662476,-47.06913658",   # Bar do Rafa
    "-22.91540157,-47.04183239",   # Bar do Saulo
    "-22.87785416,-47.05256042",   # Bar Dom Dama
    "-22.89447293,-47.07983701",   # Bar Dona Flor
    "-22.88344948,-47.06578551",   # Bar Du 6
    "-22.89323193,-47.04047310",   # Bar Lumen
    "-22.94114670,-47.02197921",   # Bar Seo Fabinho
    "-22.91881484,-47.11149841",   # Bardellas
    "-22.87831211,-47.03613638",   # Boró Botequim
    "-22.84879520,-47.07091634",   # Boteco do Pezão
    "-22.93949674,-47.09706300",   # Boteco do Wilma
    "-22.92728962,-47.05243276",   # Boteco Sem Frescura
    "-22.94004566,-47.08393431",   # Botelho's Bar
    "-22.82446457,-47.08291708",   # Botequim de Barão
    "-22.90613919,-47.05561959",   # Botiquim Pedro II
    "-22.88686696,-47.05986608",   # Buteco do Chicão

    "-22.89624722,-47.03441082",   # Cantina 85
    "-22.89211893,-47.05120764",   # Casquinha Bar
    "-22.90986772,-47.05728736",   # Cesário's Bar
    "-22.91561749,-47.04261365",   # Collab Bar
    "-22.92352296,-47.05726639",   # Deck 269
    "-22.88335698,-47.06357792",   # Edu Bar de Minas
    "-22.87893897,-46.96514632",   # Empório D' Primeira
    "-22.82797047,-47.07972838",   # Eskina Bar
    "-22.94083470,-47.05192735",   # Espaço In Bar

    "-22.91136434,-47.05647565",   # Estação Aquidabã
    "-22.94099392,-47.05684238",   # Estação Nova Europa
    "-22.88045237,-47.05041809",   # F.B.I.
    "-22.87917877,-47.06652011",   # Garten Bar
    "-22.93558976,-47.09298872",   # Loschibar
    "-22.91541224,-47.05991248",   # Maley Bistrô Bar
    "-22.92288261,-47.05689855",   # Maria Maria
    "-22.82380742,-47.08339141",   # Miúdo

    "-22.92429757,-47.05741214",   # Pirajú Botequim
    "-22.93479089,-47.08818965",   # Poeta Grill
    "-22.91397519,-47.04548141",   # Quintal do Lindomar
    "-22.89641718,-47.04555014",   # Quiosque da Horta
    "-22.88569118,-47.05791989",   # Rei do Joelho
    "-22.94247316,-47.05945804",   # Resenha dos Barbozas
    "-22.93271936,-47.06432952",   # Santa Esquina
    "-22.94080698,-47.05967982",   # Santo Espeto
    "-22.91012787,-47.08224695",   # Seo Bar
    "-22.92350723,-47.07167138",   # Seo Roma Bar e Restaurante
    "-22.87917641,-47.05035489",   # Stadium Bar
    "-22.88063850,-47.05045761",   # Vila's Bar
]

# ── Geocoding híbrido ──────────────────────────────────────────────────────────
# Converte cada entrada para o formato 'lon,lat' exigido pelo OSRM.

geolocator    = Nominatim(user_agent="roteirizacao_bares_buteco")
coords_osrm   = []
nomes_validos = []

print(f"Processando {len(dados_loc)} locais...\n")

for bar, loc in zip(bares, dados_loc):
    loc = loc.strip()
    try:
        if loc.startswith('-') and ',' in loc:
            # Coordenada direta — inverte para lon,lat (padrão OSRM)
            lat, lon = [x.strip() for x in loc.split(',')]
            coords_osrm.append(f"{lon},{lat}")
            nomes_validos.append(bar)
            print(f"✅ Coordenada direta:  {bar}")
        else:
            # Endereço de texto — geocodifica via Nominatim
            g = geolocator.geocode(loc)
            if not g and ',' in loc:
                g = geolocator.geocode(loc.split(',')[0] + ", Brasil")
            if g:
                coords_osrm.append(f"{g.longitude},{g.latitude}")
                nomes_validos.append(bar)
                print(f"✅ Geocodificado:      {bar}")
            else:
                print(f"⚠️  Não encontrado:   {bar} ({loc})")
            time.sleep(1)  # Limite obrigatório do Nominatim: 1 req/seg
    except Exception as e:
        print(f"❌ Erro em {bar}: {e}")

print(f"\n{len(nomes_validos)} locais prontos para o OSRM.")

# ── Matriz de distâncias OSRM ──────────────────────────────────────────────────
# O serviço 'table' recebe N coordenadas e retorna uma matriz N×N
# com a distância em metros percorrida por ruas entre cada par.

if len(coords_osrm) > 1:
    print("\nCalculando matriz de distâncias por ruas (OSRM)...")
    coords_str = ";".join(coords_osrm)
    url        = f"http://router.project-osrm.org/table/v1/driving/{coords_str}?annotations=distance"

    response = requests.get(url)
    data     = response.json()

    if data.get('code') == 'Ok':
        # Converte metros → quilômetros
        matriz_km = [[round(m / 1000, 2) for m in linha] for linha in data['distances']]
        df_matriz = pd.DataFrame(matriz_km, index=nomes_validos, columns=nomes_validos)
        df_matriz.to_excel("matriz_distancias_bares.xlsx")
        print("\n✅ Arquivo 'matriz_distancias_bares.xlsx' gerado com sucesso!")
        display(df_matriz)
    else:
        print("❌ Erro na resposta do OSRM:", data.get('message', 'desconhecido'))
else:
    print("Não há coordenadas suficientes para gerar a matriz.")

Processando 57 locais...

✅ Coordenada direta:  Ponto de Saida
✅ Coordenada direta:  Bar da Loira
✅ Coordenada direta:  Bar da Vila
✅ Coordenada direta:  Bar do Baú
✅ Coordenada direta:  Bar do Braz
✅ Coordenada direta:  Bar do Cação
✅ Coordenada direta:  Bar do Carioca
✅ Coordenada direta:  Bar do Furlan
✅ Coordenada direta:  Bar do Léo
✅ Coordenada direta:  Bar do Luís
✅ Coordenada direta:  Bar do Nicola
✅ Coordenada direta:  Bar do Português
✅ Coordenada direta:  Bar do Rafa
✅ Coordenada direta:  Bar do Saulo
✅ Coordenada direta:  Bar Dom Dama
✅ Coordenada direta:  Bar Dona Flor
✅ Coordenada direta:  Bar Du 6
✅ Coordenada direta:  Bar Lumen
✅ Coordenada direta:  Bar Seo Fabinho
✅ Coordenada direta:  Bardellas
✅ Coordenada direta:  Boró Botequim
✅ Coordenada direta:  Boteco do Pezão
✅ Coordenada direta:  Boteco do Wilma
✅ Coordenada direta:  Boteco Sem Frescura
✅ Coordenada direta:  Botelho's Bar
✅ Coordenada direta:  Botequim de Barão
✅ Coordenada direta:  Botiquim Pedro II
✅ Coorde

,Ponto de Saida,Bar da Loira,Bar da Vila,Bar do Baú,Bar do Braz,Bar do Cação,Bar do Carioca,Bar do Furlan,Bar do Léo,Bar do Luís,...,Quintal do Lindomar,Quiosque da Horta,Rei do Joelho,Resenha dos Barbozas,Santa Esquina,Santo Espeto,Seo Bar,Seo Roma Bar e Restaurante,Stadium Bar,Vila's Bar
Ponto de Saida,0.00,5.79,5.78,3.02,4.80,4.34,1.99,1.11,3.39,3.25,...,3.95,4.55,2.96,6.24,5.12,6.00,1.58,3.79,4.17,3.92
Bar da Loira,5.11,0.00,9.22,3.43,2.59,9.77,5.15,6.37,2.58,3.92,...,8.06,8.90,7.98,6.23,5.67,6.25,4.09,4.45,9.59,9.35
Bar da Vila,5.98,9.17,0.00,8.28,8.10,7.03,6.89,5.07,9.54,5.72,...,2.13,3.94,5.96,5.24,5.18,5.12,7.56,6.42,6.86,6.93
Bar do Baú,2.67,4.23,7.38,0.00,1.99,7.00,3.53,3.77,2.11,2.08,...,6.22,7.06,5.62,5.29,4.59,5.31,1.44,2.61,6.83,6.59
Bar do Braz,4.24,2.54,8.01,1.68,0.00,8.58,5.11,5.35,2.75,2.70,...,6.85,7.69,7.20,5.11,4.55,5.14,3.02,3.24,8.40,8.16
Bar do Cação,4.55,10.14,5.94,7.37,9.07,0.00,4.22,3.28,7.29,6.69,...,4.26,2.81,1.37,8.21,7.33,7.97,5.56,7.23,0.40,0.16
Bar do Carioca,2.13,6.06,7.03,4.06,6.84,4.74,0.00,1.69,3.66,4.50,...,5.20,4.57,3.03,7.49,6.37,7.25,2.59,5.04,4.57,4.33
Bar do Furlan,2.00,7.32,4.85,4.55,6.25,3.76,2.28,0.00,4.81,3.87,...,3.44,2.95,2.37,6.03,5.14,5.79,3.00,4.41,3.59,3.34
Bar do Léo,3.98,2.88,9.25,2.88,3.65,8.17,3.55,4.71,0.00,4.53,...,7.42,8.02,6.38,7.79,7.23,7.82,2.98,5.07,8.00,7.75
Bar do Luís,3.56,3.99,5.67,2.22,2.92,8.07,4.43,3.90,4.49,0.00,...,4.51,5.35,7.00,3.44,3.04,4.40,2.34,0.54,7.90,7.97


---
## 🚗 Etapa 4 — Otimização do Roteiro com Velocidade Constante

Resolve o **Problema do Caixeiro Viajante (VRP)** com OR-Tools, considerando:

- 🗓️ **7 veículos = 7 dias da semana** — cada dia é uma rota independente
- 🕐 **Janelas de tempo**: visita o bar apenas quando está aberto, com mínimo de 30 min de permanência
- 🚦 **Velocidade fixa de 30 km/h** em todos os trechos, independentemente do dia ou horário
- 🎯 **Objetivo**: minimizar a distância total percorrida
- ⏳ O otimizador roda por até 1000 segundos buscando a melhor rota
- 📈 A cada solução melhor encontrada, o tempo decorrido e a distância total são impressos no log de execução

> 📁 **Arquivos necessários** neste diretório:
> - `matriz_distancias_bares.xlsx` — gerado na Etapa 3
> - `hora.csv` — horários dos bares (sep=`;`), colunas: `nome`, `segunda-abre`, `segunda-fecha`, ...


In [ ]:
import time
import pandas as pd
import numpy as np
from ortools.constraint_solver import routing_enums_pb2, pywrapcp

# ── 1. Carregar os dados ───────────────────────────────────────────────────────

df_dist       = pd.read_excel("matriz_distancias_bares.xlsx", index_col=0)
df_hora       = pd.read_csv("butecos_campinas.csv", sep=";") # FIX: Changed separator to semicolon and added correct path

# Limpeza de nomes para garantir sincronização entre as tabelas
df_hora['nome'] = df_hora['nome'].str.strip()
df_dist.index   = df_dist.index.str.strip()
df_dist.columns = df_dist.columns.str.strip()

# Corrige possíveis vírgulas decimais na matriz (formato pt-BR)
for col in df_dist.columns:
    df_dist[col] = df_dist[col].astype(str).str.replace(',', '.', regex=False).astype(float)

dist_matrix       = df_dist.values
node_names        = list(df_dist.index)
depot_idx         = node_names.index("Ponto de Saida")  # Índice do ponto de partida/retorno
num_nodes         = len(dist_matrix)
num_vehicles      = 7   # Um 'veículo' por dia da semana
tempo_permanencia = 30  # Minutos mínimos obrigatórios em cada bar

print(f"📊 {num_nodes} nós | Ponto de saída: índice {depot_idx} ('{node_names[depot_idx]}')")


# ── 2. Velocidade fixa de deslocamento ──────────────────────────────────────────
# Velocidade constante de 30 km/h para todos os trechos, em qualquer dia/horário.

VELOCIDADE_FIXA_KMH = 30.0

def get_velocidade(dia_nome, minutos):
    """Retorna a velocidade (km/h) fixa, independente de dia/horário."""
    return VELOCIDADE_FIXA_KMH

def str_to_minutes(time_str):
    """Converte 'HH:MM' para minutos desde meia-noite. Retorna -1 se inválido."""
    if pd.isna(time_str) or not isinstance(time_str, str):
        return -1
    h, m = map(int, time_str.split(':'))
    return h * 60 + m

# FIX: Added DIA_MAP dictionary
DIA_MAP = {
    "segunda": "segunda",       "segunda-feira": "segunda",
    "terça": "terça",           "terca": "terça",
    "terça-feira": "terça",     "terca-feira": "terça",
    "quarta": "quarta",         "quarta-feira": "quarta",
    "quinta": "quinta",         "quinta-feira": "quinta",
    "sexta": "sexta",           "sexta-feira": "sexta",
    "sábado": "sábado",         "sabado": "sábado",
    "domingo": "domingo",
}

# ── 3. Janelas de tempo e velocidades por nó e por dia ────────────────────────
# Para cada combinação dia × bar, definimos:
#   - time_windows: janela [abre, fecha - 30min] em que o bar pode ser visitado
#   - node_speeds_per_day: velocidade de saída estimada para aquele horário

dias_semana         = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado', 'Domingo']
time_windows        = {}
node_speeds_per_day = {}

for v in range(num_vehicles):
    time_windows[v]        = {}
    node_speeds_per_day[v] = {}
    dia_str = dias_semana[v]
    # Use DIA_MAP to normalize dia_key to match CSV column names
    dia_key = DIA_MAP.get(dia_str.lower(), dia_str.lower())

    # Dicionário para traduzir os nomes da sua lista para os nomes reais do CSV
    correcao_nomes = {
        "Poco Loco": "1 Poco Loco",
        "Bar do Baú": "Bar do Bau",
        "Bar do Luís": "Bar do Luis",
        "Botelho's Bar": "Botelho’s Bar",
        "Cesário's Bar": "Cesario’s Bar",
        "Empório D' Primeira": "Empório D’ Primeira",
        "Vila's Bar": "Vila’s Bar"
    }

    for i, nome in enumerate(node_names):
        if i == depot_idx:
            time_windows[v][i] = (0, 1680)
            # Horário de saída: 18h em dias úteis, 12h no fim de semana
            inicio_role = 18 * 60 if v < 5 else 12 * 60
            node_speeds_per_day[v][i] = get_velocidade(dia_str, inicio_role)
            continue

        # Verifica se o nome precisa ser "traduzido" antes de buscar no CSV
        nome_busca = correcao_nomes.get(nome, nome);

        row = df_hora[df_hora['nome'] == nome_busca]

        abre  = str_to_minutes(row[f'{dia_key}-abre'].values[0])  if not row.empty else -1
        fecha = str_to_minutes(row[f'{dia_key}-fecha'].values[0]) if not row.empty else -1

        if abre != -1:
            if fecha <= abre:
                fecha += 1440  # Bar fecha após meia-noite
            time_windows[v][i]        = (abre, max(abre, fecha - tempo_permanencia))
            node_speeds_per_day[v][i] = get_velocidade(dia_str, abre)
        else:
            time_windows[v][i]        = (-1, -1)  # Bar fechado neste dia
            node_speeds_per_day[v][i] = 1.0       # Valor nulo, não será usado


# ── 4. Modelo OR-Tools ─────────────────────────────────────────────────────────

manager = pywrapcp.RoutingIndexManager(num_nodes, num_vehicles, depot_idx)
routing = pywrapcp.RoutingModel(manager)

# Um callback de tempo por veículo (cada dia tem velocidades diferentes)
time_callback_indices = []
for v in range(num_vehicles):
    speeds = node_speeds_per_day[v]

    def create_callback(speeds_dict):
        def _callback(from_index, to_index):
            from_node = manager.IndexToNode(from_index)
            to_node   = manager.IndexToNode(to_index)
            dist_km   = dist_matrix[from_node][to_node]
            v_kmh     = speeds_dict[from_node]
            tempo     = int(np.ceil((dist_km / v_kmh) * 60))  # minutos
            if from_node != depot_idx:
                return tempo + tempo_permanencia  # adiciona permanência ao sair do bar
            return tempo
        return _callback

    cb_idx = routing.RegisterTransitCallback(create_callback(speeds))
    time_callback_indices.append(cb_idx)

# Dimensão de tempo — 1680 min = 28h (janela máxima do dia)
routing.AddDimensionWithVehicleTransits(
    time_callback_indices, 1680, 1680, False, 'Time'
)
time_dimension = routing.GetDimensionOrDie('Time')

# Callback de distância — objetivo principal: minimizar km percorridos
dist_cb_idx = routing.RegisterTransitCallback(
    lambda i, j: int(dist_matrix[manager.IndexToNode(i)][manager.IndexToNode(j)] * 100)
)
routing.SetArcCostEvaluatorOfAllVehicles(dist_cb_idx)

# Restrições de janela de tempo por nó e por veículo
for node in range(num_nodes):
    if node == depot_idx:
        continue
    index       = manager.NodeToIndex(node)
    veiculos_ok = []
    for v in range(num_vehicles):
        abre, fecha = time_windows[v][node]
        if abre != -1:
            veiculos_ok.append(v)
            cond = routing.VehicleVar(index) == v
            routing.solver().Add(time_dimension.CumulVar(index) >= abre * cond)
            routing.solver().Add(time_dimension.CumulVar(index) <= fecha + 1680 * (1 - cond))
    veiculos_ok.append(-1)  # Permite não visitar o nó em último caso
    routing.VehicleVar(index).SetValues(veiculos_ok)
    routing.AddDisjunction([index], 10_000_000)  # Penalidade alta para não pular bares


# ── 5. Acompanhar a evolução da busca ──────────────────────────────────────────
# Sempre que o otimizador encontra uma solução melhor que a anterior, imprime
# o tempo decorrido e a distância total naquele momento.

inicio_busca      = time.time()
contador_solucoes = [0]
melhor_distancia  = [float('inf')]

def mostrar_evolucao():
    contador_solucoes[0] += 1
    distancia_atual = routing.CostVar().Value() / 100  # desfaz a escala usada no callback de distância
    if distancia_atual < melhor_distancia[0]:
        melhor_distancia[0] = distancia_atual
        decorrido = time.time() - inicio_busca
        print(f"🔄 Melhora #{contador_solucoes[0]:>3} | {decorrido:7.1f}s decorridos | Distância total: {distancia_atual:8.2f} km")

routing.AddAtSolutionCallback(mostrar_evolucao)


# ── 6. Resolver ────────────────────────────────────────────────────────────────

search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = 1000

print(f"\n⏳ Calculando roteiro semanal (limite de {search_parameters.time_limit.seconds}s)...\n")
solution = routing.SolveWithParameters(search_parameters)

if solution:
    tempo_total = time.time() - inicio_busca
    print(f"\n✅ Busca concluída em {tempo_total:.1f}s | {contador_solucoes[0]} soluções avaliadas | Melhor distância: {melhor_distancia[0]:.2f} km")


# ── 7. Exportar resultado ──────────────────────────────────────────────────────

if solution:
    dropped_nodes = []
    for node in range(routing.Size()):
        if routing.IsStart(node) or routing.IsEnd(node):
            continue
        if solution.Value(routing.NextVar(node)) == node:
            dropped_nodes.append(manager.IndexToNode(node))

    print(f"\n🚫 Bares deixados de fora pelo otimizador: {[node_names[i] for i in dropped_nodes]}\n")
    route_data    = []
    total_dist    = 0
    total_visited = 0

    for v in range(num_vehicles):
        index = routing.Start(v)
        dia   = dias_semana[v]
        ordem = 1

        if routing.IsEnd(solution.Value(routing.NextVar(index))):
            continue  # Nenhum bar visitado neste dia

        while not routing.IsEnd(index):
            node_idx   = manager.IndexToNode(index)
            t_min      = solution.Min(time_dimension.CumulVar(index))
            h, m       = divmod(t_min, 60)
            nome_local = "Ponto de Saída" if node_idx == depot_idx else node_names[node_idx]
            velocidade = node_speeds_per_day[v][node_idx]

            route_data.append({
                "Dia da Semana":     dia,
                "Ordem":             ordem,
                "Local":             nome_local,
                "Horário Chegada":   f"{int(h) % 24:02d}:{int(m):02d}",
                "Velocidade (km/h)": f"{velocidade} km/h",
            })

            previous_index = index
            index = solution.Value(routing.NextVar(index))
            total_dist += dist_matrix[manager.IndexToNode(previous_index)][manager.IndexToNode(index)]
            ordem += 1
            if node_idx != depot_idx:
                total_visited += 1

        # Registra o retorno ao ponto de saída ao final do dia
        node_idx = manager.IndexToNode(index)
        t_min    = solution.Min(time_dimension.CumulVar(index))
        h, m     = divmod(t_min, 60)
        route_data.append({
            "Dia da Semana":     dia,
            "Ordem":             ordem,
            "Local":             "Retorno ao Ponto",
            "Horário Chegada":   f"{int(h) % 24:02d}:{int(m):02d}",
            "Velocidade (km/h)": "-",
        })

    df_resultado = pd.DataFrame(route_data)
    df_resultado.to_csv("roteiro_semanal.csv", index=False, encoding="utf-8-sig", sep=";")

    print(f"✅ Roteiro gerado! Salvo como 'roteiro_semanal.csv'")
    print(f"📍 Bares visitados: {total_visited} | 🛣️  Distância total: {total_dist:.2f} km")
    display(df_resultado)
else:
    print("❌ Nenhuma solução encontrada. Verifique os horários em 'hora.csv'.")
